In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import qutip as q

# Quantum circuits with QuTiP

[QuTiP](http://qutip.org/), the _Quantum Toolbox in Python_, is a great package that often comes in handy for quick calculations on quantum optical systems and quantum information processing. The exercises you did last time, for example, would be very easy to do with QuTiP. It is mainly designed to simulate open quantum systems, but to enable this goal it also includes a small library of quantum circuit tools (which is somewhat rudimentary and  incomplete, but it does give us some help).

To find out all about what the package can do, check the [documentation](http://qutip.org/docs/latest/index.html) and [tutorials](http://qutip.org/tutorials.html). For now, let's just dig in. 

The core object in QuTiP is the `QObj`, which can represent state vectors (ket/bra) or operators like density matrices and unitary operators. As examples, let's create the $|0\rangle$ ket, the same bra, and $\sigma_x$ operator:

In [ ]:
q.Qobj(np.array([1, 0]))

In [ ]:
q.Qobj(np.array([[1, 0]]))

In [ ]:
q.Qobj(np.array([[0, 1], [1, 0]]))

Note the information about the objects noted above their nicely formatted representations.

There are easier ways of creating these basic objects using built-in functions:

In [ ]:
ket0 = q.basis(2, 0)
ket0

In [ ]:
ket0.dag()  # dagger operator, which for kets convert to bra

In [ ]:
q.sigmax()

For multi-qubit systems, you combine states and operators into higher-dimensional Hilbert spaces using `tensor`:

In [ ]:
q.tensor(ket0, ket0)

In [ ]:
q.tensor(q.sigmax(), q.identity(2))

An alternative shorthand is the `&` operator:

In [ ]:
q.sigmax() & q.identity(2)

A few useful features - calculation of expectation values, built-in Bell states, partial traces, fidelities:

In [ ]:
(q.expect(q.sigmax(), ket0), 
 q.expect(q.sigmay(), ket0),
 q.expect(q.sigmaz(), ket0))

In [ ]:
Psi_minus = q.bell_state('11')
Psi_minus

In [ ]:
Psi_minus.ptrace(0)

In [ ]:
def W(p):
    """Werner mixed state"""
    return p / 4 * q.tensor(q.identity(2), q.identity(2)) + (1 - p) * q.ket2dm(Psi_minus)

In [ ]:
W(1/3)

In [ ]:
q.fidelity(W(1/3), Psi_minus)

In [ ]:
ps = np.linspace(0, 1, 100)
fidelities = [q.fidelity(W(p), Psi_minus) for p in ps]

plt.plot(ps, fidelities)
plt.xlabel(r'$p$')
plt.ylabel('fidelity')

### Quantum information processing

Many other gates also exist as built-ins. Note that the Hadamard gate is called SNOT (or `hadamard_transform`):

In [ ]:
q.gates.snot()

In [ ]:
q.gates.hadamard_transform(1)

In [ ]:
q.gates.cnot()

In [ ]:
q.gates.phasegate(np.pi / 4)

In [ ]:
q.gates.t_gate()

In [ ]:
q.gates.toffoli()

However, since [QuTiP 5.0](https://arxiv.org/abs/2412.04705), most of the quantum circuit simulation tools have been moved into a separate sub-package, [qutip-qip](https://qutip-qip.readthedocs.io/en/stable/):

In [ ]:
from qutip_qip.circuit import QubitCircuit
import qutip_qip.operations as ops

With this package, more complex gates are possible, including multi-qubit controlled gates:

In [ ]:
ops.cnot(control=1, target=0)

In [ ]:
ops.controlled_gate(ops.y_gate(), controls=[1], targets=[2], N=3, control_value=0)  # controlled-Z with control value of 0 instead of 1 as usual

However, the primary way to use the package for quantum circuit simulation (at the gate level) is to use the `QubitCircuit` object:

In [ ]:
circuit = QubitCircuit(N=2)

You can add gates, display the circuit diagram, and get the individual propagation matrices for your circuit:

In [ ]:
circuit.add_gate('SNOT', targets=1)
circuit.add_gate('CNOT', targets=0, controls=1)
circuit.add_gate('CRZ', targets=1, controls=0, arg_value=-np.pi)  # controlled=Z

In [ ]:
circuit

In [ ]:
circuit.gates

In [ ]:
circuit.propagators()

Note that some gates require argument values (like rotation angle for RZ). Some require controls and/or targets.

To know which gates are available, you can look at the [documentation](https://qutip-qip.readthedocs.io/en/latest/qip-basics.html#gates) or run this:

In [ ]:
for k, v in ops.GATE_CLASS_MAP.items():
    print(f'{k:>15} : {v.__doc__.strip().splitlines()[0]}')

To calculate the overall unitary propagation matrix for the circuit, use `gate_sequence_product`:

In [ ]:
U = ops.gate_sequence_product(circuit.propagators())
U

We can now apply the circuit to the input states. In this case we initialize the inputs in the state $|1\rangle |1\rangle$. The circuit then prepares the $|\Psi^-\rangle$ Bell state:

In [ ]:
ket0 = q.basis(2, 0)
input_states = ket0 & ket0

In [ ]:
U * input_states

Or, using the package as intended, by running the circuit itself:

In [ ]:
circuit.run(state=input_states)

For more detailed usage, refer to the [documentation](https://qutip-qip.readthedocs.io/en/latest/qip-simulator.html).